In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the cleaned data
print("Loading cleaned dataset...")
# Adjust this path if you saved it somewhere else (like just 'cleaned_data.csv')
df = pd.read_csv('../data/cleaned_data.csv')

# Drop any rows where cleaned_content became NaN during saving/loading
df = df.dropna(subset=['cleaned_content'])

X = df['cleaned_content']
y = df['sentiment']

# 2. Vectorization (Convert Text to Numbers using TF-IDF)
print("Vectorizing text data...")
# We use max_features=5000 to keep the model lightweight and fast for streaming
vectorizer = TfidfVectorizer(max_features=5000)
X_vectorized = vectorizer.fit_transform(X)

# 3. Train/Test Split (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

# 4. Train Model 1: Naive Bayes
print("\nTraining Naive Bayes Model...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)
nb_accuracy = accuracy_score(y_test, nb_predictions)
print(f"Naive Bayes Accuracy: {nb_accuracy * 100:.2f}%")

# 5. Train Model 2: Logistic Regression
print("\nTraining Logistic Regression Model (this might take a minute)...")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f"Logistic Regression Accuracy: {lr_accuracy * 100:.2f}%")

# 6. Save the Best Model and the Vectorizer for Member 2
# Let's assume Logistic Regression performs better (it usually does for TF-IDF)
best_model = lr_model if lr_accuracy >= nb_accuracy else nb_model
best_model_name = "Logistic Regression" if lr_accuracy >= nb_accuracy else "Naive Bayes"

print(f"\nWinner: {best_model_name}! Saving files for the Spark Pipeline...")

# Create the models folder if it doesn't exist
os.makedirs('../models/', exist_ok=True)

# Save the files
joblib.dump(best_model, '../models/sentiment_model.pkl')
joblib.dump(vectorizer, '../models/vectorizer.pkl')

print("Success! 'sentiment_model.pkl' and 'vectorizer.pkl' saved in the models/ folder.")

Loading cleaned dataset...
Vectorizing text data...

Training Naive Bayes Model...
Naive Bayes Accuracy: 83.06%

Training Logistic Regression Model (this might take a minute)...
Logistic Regression Accuracy: 86.46%

Winner: Logistic Regression! Saving files for the Spark Pipeline...
Success! 'sentiment_model.pkl' and 'vectorizer.pkl' saved in the models/ folder.
